# Playground for code

In [ ]:
a = [1,2,3,4,5,6,7,8,9,10]

In [ ]:
import random

random.seed(42)
random.shuffle(a)
print(a)

## Lookup parameters in runs

In [ ]:
from pathlib import Path
import pandas as pd
data_prod_path = "../nn_runs/"
base = Path(data_prod_path)

# find directories with "log" in their name (case-insensitive)
log_dirs = [p for p in base.rglob('*') if p.is_dir() and 'log' in p.name.lower()]

# include base if its name contains "log"
if 'log' in base.name.lower():
    log_dirs.append(base)

# gather all .txt files from those directories
log_files = sorted({f for d in log_dirs for f in d.glob('*.txt')})

# read contents into a dict {filepath: text}
log_texts = {str(p): p.read_text(encoding='utf-8', errors='replace') for p in log_files}
# convert to DataFrame for easy viewing
log_df = pd.DataFrame([{'path': path, 'name': Path(path).name, 'text': text} for path, text in log_texts.items()])
log_df['size'] = log_df['text'].str.len()
log_df['preview'] = log_df['text'].str.slice(0, 200)
log_df = log_df.sort_values('path').reset_index(drop=True)
log_df

print(f"Found {len(log_texts)} .txt files in directories with 'log' in their name.")
for path in log_texts:
    print(path)

## Collect iterres extract metrics

In [ ]:
from collect_iterres import extract_metrics_from_log
from paths import path_to_nn_runs
metrics, run_info =extract_metrics_from_log(f"{path_to_nn_runs}/iter_excl_PFI_parallel_n500_k12/cluster_b4_p10_run1/log_run1.txt")

In [ ]:
run_info

In [ ]:
metrics

In [ ]:
from paths import data_prod_path, path_to_nn_runs
import pandas as pd

df = pd.read_csv(f"{data_prod_path}/bact_clusters_with_genus.csv", index_col=0)
df_unsorted = df.copy().head(5)
df_sorted = df.sort_values(by="Cluster", ascending=False).groupby(['genus'])
df = df_sorted.head(5)

#display(df)
display(df_sorted.all())

In [ ]:
import random
top_kmer_df = pd.read_csv(f"{path_to_nn_runs}/iter_excl_PFI_parallel_n500_k12/cluster_b3_p12_run1/top_interaction_pair_kmers.csv", index_col=0)
top_kmer_df['PFI'] = [random.random() for _ in range(len(top_kmer_df))]
top_kmer_df['folder'] = "cluster_b3_p12_run1"
display(top_kmer_df)
print(f"Number of uniques in entity: {top_kmer_df['entity'].nunique()}")
print(f"Uniques in entity: {sorted(top_kmer_df['entity'].unique().tolist())}")

In [ ]:
from collect_iterres import balanced_top_k
sort_by = 'PFI'
top_kmers = 500
bact_kmers_df = balanced_top_k(
                df=top_kmer_df,
                group_cols=['folder', 'entity'],
                sort_col=sort_by,
                total_k=top_kmers
            )

display(bact_kmers_df)
print(f"Number of uniques in entity: {bact_kmers_df['entity'].nunique()}")
print(f"Uniques in entity: {sorted(bact_kmers_df['entity'].unique().tolist())}")

## Load PFI objects for structure

In [ ]:
from paths import path_to_nn_runs
import pandas as pd
import joblib
expected_interactions = joblib.load(f"{path_to_nn_runs}/IterExcl_encoded_sketches_n500_k12/cluster_b0_p0_run1/pfi_objects_encoded_sketches_n500_k12/expected_interactions.jbl")

In [ ]:
for ent in expected_interactions:
    print(ent)

## Check hostrange 

In [ ]:
from paths import raw_data_path
from io_operations import call_hostrange_df
from manipulations import binarize_host_range, hostrange_df_to_dict
import os

data2_flag = False
if data2_flag:
    bact_lookup, host_range_df = call_hostrange_df(os.path.join(raw_data_path, "phagehost_KU/data2_EOP.xlsx"), sheet_name="Sheet1", data2=True)
else:
    bact_lookup, host_range_df = call_hostrange_df(os.path.join(raw_data_path, "phagehost_KU/Hostrange_data_all_crisp_iso.xlsx"), data2=False)
host_range_data = binarize_host_range(hostrange_df_to_dict(host_range_df), continous=False)
host_range_data = {bact.replace("_reoriented", ""): interactions for bact, interactions in host_range_data.items()} # if "_reoriented" is in the bacteria names in host_range_data, remove it to match the bacteria names in the presence matrix.

host_range_data

# Evaluate the compotutional expense of summing over all interaction and occurence values in pfi folders

In [ ]:
import os
import joblib
pfi_dir_example = "/home/projects/s215045/PredictPhagePPI/nn_runs/FFNN_Test_data2/cluster_bX_pX_run1/pfi_objects_encoded_sketches_data2_n500_k12"

print(os.listdir(pfi_dir_example))
int_pairs = joblib.load(os.path.join(pfi_dir_example, "interaction_pairs.jbl"))
normalized_scores = joblib.load(os.path.join(pfi_dir_example, "normalized_interaction_rate.jbl"))
print(int_pairs[0:4])

In [ ]:
print(f"Number of interaction pairs: {len(normalized_scores)}")
for i, pair in enumerate(normalized_scores.items()):
    print(f"{i+1}: {pair}")
    if i >= 20:
        break

In [ ]:
print(list(int_pairs.values())[0:4])
sum(int_pairs.values())